# RFDC V4 -- Row-Based Pulse Program Notebook

This notebook programs the pump / spectroscopy / detection sequence from a **flat list of
sequencer rows**, where each row carries its own `gap_s`, and for TX lanes the *actual
waveform array* (not a duration you have to keep in sync by hand). `mask` and `DUR0..DUR3`
are always derived from what's actually in the row -- they are never typed separately, so
there is nothing to accidentally desynchronize.

**Control confirmed: Pulse_Sequencer only, no `TX_Multi_Gate` internal segment-table mode.**
Every TX row writes a nonzero `DUR0`/`DUR1` for whichever lane fires -- `TX_Multi_Gate` reads
that as `hw_active_length != 0`, which is EXTERNAL mode: open one window sized exactly by that
duration, then close. `TXM_NUM_SEGMENTS`/`TXM_SEG_SEL`/`TXM_ACTIVE_LEN` (the internal-table
registers) are only ever *defined* in the register map cell and never written anywhere in this
notebook -- multi-pulse patterns like the probe train are one continuous waveform from
`chirp_train()`/`waveform()` compiled into a single row, not a gate-side segment table. This
was true before this revision too; nothing needed to change here.

**What changed in this revision:** the Ethernet "overlap" in the previous draft computed an
inter-run wait but then blocked on the full transfer anyway before sleeping that wait on top --
so nothing was actually overlapping. RX is now double-buffered and a background sender thread
owns the socket, so a shot's send genuinely runs concurrently with the next shot's hardware.
See Cell 12/13/14/16. Also added: `dump_state()` for one-call hardware diagnostics, called
automatically on any run-loop exception, plus a `DEBUG_VERBOSE` flag for extra per-event
logging (Cell 2 config).

This mirrors, and is meant to be a drop-in replacement for, the row-programming and DMA/loop
machinery already validated in `RFDC_V4_Pump_Pi2_Probe_TTL_Loop_Qualification_Ethernet.ipynb`
(same register map, same `AXIS_BEAT_HZ`/`SEQ_CLK_HZ` constants, same length-prefixed Ethernet
protocol). Nothing here requires a new bitstream beyond the 256-row `Pulse_Sequencer`.

**Status: design draft, not yet run against hardware.** Two things are called out explicitly
below as needing benchmarking before you trust the numbers: (1) how long a PYNQ
`transfer()` call takes to issue for a refill, and (2) real achieved Ethernet Mbps on this
board's link. Everything else follows directly from the VHDL and the existing qualification
notebook's proven register sequences.


## Cell 1 — Overlay + imports (only this)

Kept as its own cell since it's the slow step and rarely needs re-running once the board is up.


In [1]:
# ================= OVERLAY + IMPORTS (run this once) =================
import time, socket, json, struct, threading, queue
import numpy as np
import xrfclk
import xrfdc
from pynq import Overlay, allocate

BITFILE = "./final.bit"   # confirm this is the 256-row Pulse_Sequencer build
base = Overlay(BITFILE)
print("Overlay loaded:", BITFILE)


Overlay loaded: ./final.bit


## Cell 2 — User configuration

Board/clock constants are fixed by the hardware; everything else here is yours to edit.


In [2]:
# ================= USER CONFIG =================
AXIS_BEAT_HZ = 15.36e6          # DAC/ADC-side fabric clock (fixed by clock wizard)
SAMPLES_PER_BEAT = 8            # samples per AXIS beat (fixed by IP width)
SEQ_CLK_HZ = 99_999_985.0       # Pulse_Sequencer's own clock (fixed, ~100 MHz)
FS_HZ = 122.88e6                # DAC/ADC sample rate
AMPLITUDE = 32760               # int16 full-scale headroom for I/Q waveforms

DAC_A_NCO_MHZ = 10.0
DAC_B_NCO_MHZ = 0.2

DMA_MAX_BYTES = (1 << 26) - 1   # 26-bit simple-DMA length register limit (~67.1 MB)
CAP_MAX_S = 0.260               # RX DMA capacity ceiling at 16-bit real samples
TX_MAX_S = 0.136                # TX DMA capacity ceiling at complex 8i8q

# Ethernet / PC handshake
PC_HOST = "192.168.3.136"       # PC running the companion receiver notebook
PC_PORT = 5001
ETH_TIMEOUT_S = 15.0
# MEASURE THIS on your actual link -- the qualification notebook measured ~180-200 Mbps
# today, ~950 Mbps once the PS GEM clocking fix lands. Leave conservative (low) until you
# have a real number from this board/network. This is now purely informational (see Cell 13)
# since the double-buffered sender below measures real overlap live instead of assuming it.
ETHERNET_LINK_MBPS_MEASURED = 190
ETH_MARGIN_S = 5e-3             # safety margin added on top of the raw transfer-time estimate

# MEASURE THIS: how long dma.sendchannel.transfer(buf) takes to issue from Python for a
# refill-sized buffer on this board. Used as a safety check against each refill row's gap_s
# so the notebook can warn if a refill genuinely won't fit in its window.
TRANSFER_CALL_OVERHEAD_S_ESTIMATE = 1e-3

# Prints extra diagnostic lines during the run: refill events, buffer-reuse waits, background
# sender status. Turn off once you trust the timing and just want the compact per-shot line.
DEBUG_VERBOSE = True

EXPECTED_IP_PATHS = {
    "sequencer": "radio/AXI_Pulse_Sequencer_0",
    "tx_gate_b": "radio/AXI_TX_Multi_Gate_0",
    "tx_gate_a": "radio/AXI_TX_Multi_Gate_1",
    "cap_gate_b": "radio/receiver/channel_20/AXI_Capture_Gate_0",
    "cap_gate_a": "radio/receiver/channel_21/AXI_Capture_Gate_0",
    "rx_dma_b": "radio/receiver/channel_20/axi_dma_real",
    "rx_dma_a": "radio/receiver/channel_21/axi_dma_real",
    "tx_dma_b": "radio/axi_dma_dac_0",
    "tx_dma_a": "radio/axi_dma_dac_1",
}
print("Config loaded.")


Config loaded.


## Cell 3 — IP resolution

Same pattern as the qualification notebook: fail loudly if the loaded `.hwh` doesn't match
the expected A/B topology, rather than silently resolving the wrong IP.


In [3]:
# ================= IP RESOLUTION =================
def get_by_path(root, path):
    obj = root
    for part in path.split('/'):
        obj = getattr(obj, part)
    return obj

missing = [p for p in EXPECTED_IP_PATHS.values() if p not in base.ip_dict]
if missing:
    print("Missing expected HWH paths:")
    for p in missing:
        print("  ", p)
    raise KeyError("The loaded .hwh does not match the validated A/B topology.")

seq   = get_by_path(base, EXPECTED_IP_PATHS['sequencer'])
tx_b  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_b'])
tx_a  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_a'])
cap_b = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_b'])
cap_a = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_a'])
dma_rb = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_b'])
dma_ra = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_a'])
dma_tb = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_b'])
dma_ta = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_a'])

rfdc = get_by_path(base, 'radio/rfdc')
dac_b = rfdc.dac_tiles[0].blocks[0]
dac_a = rfdc.dac_tiles[2].blocks[0]
adc_b = rfdc.adc_tiles[2].blocks[0]
adc_a = rfdc.adc_tiles[2].blocks[1]
print('IP resolved.')


IP resolved.


## Cell 4 — Register map + unit-conversion helpers

Unchanged from the qualification notebook -- these are the ground-truth hardware constants.


In [4]:
# ================= REGISTER MAP =================
SEQ_ROW_SEL=0x00; SEQ_MASK=0x04; SEQ_GAP=0x08
SEQ_DUR0=0x0C; SEQ_DUR1=0x10; SEQ_DUR2=0x14; SEQ_DUR3=0x18
SEQ_COMMIT=0x1C; SEQ_TABLE_LEN=0x20; SEQ_ENABLE=0x24
SEQ_RESET=0x28; SEQ_ROW_PTR=0x2C; SEQ_CDC_OVERRUN=0x30
SEQ_TTL_ARM=0x34; SEQ_TTL_MODE=0x38; SEQ_TTL_STATUS=0x3C
SEQ_TTL_STATUS_CLEAR=0x40; SEQ_TTL_EDGE_COUNT=0x44
TTL_ARMED=1<<0; TTL_RUNNING=1<<1; TTL_RUN_DONE=1<<2

TXM_SEG_SEL=0x00; TXM_ACTIVE_LEN=0x04; TXM_GAP_LEN=0x08; TXM_COMMIT=0x0C
TXM_NUM_SEGMENTS=0x10; TXM_STATUS=0x1C
TX_BUSY=1<<0; TX_OVERRUN=1<<1

CAP_LENGTH=0x00; CAP_STATUS=0x08; CAP_CLEAR=0x0C
CAP_BUSY=1<<0; CAP_OVERFLOW=1<<1

DMASR_HALTED=1<<0; DMASR_IDLE=1<<1; DMASR_ERR_MASK=(1<<4)|(1<<5)|(1<<6)

LANE_TX_B, LANE_TX_A, LANE_RX_B, LANE_RX_A = 0, 1, 2, 3
LANE_BIT = {"TX_B": LANE_TX_B, "TX_A": LANE_TX_A, "RX_B": LANE_RX_B, "RX_A": LANE_RX_A}
TX_LANES = ("TX_A", "TX_B")
RX_LANES = ("RX_A", "RX_B")

def beats_for(seconds):
    return max(1, int(round(seconds * AXIS_BEAT_HZ)))

def samples_for(beats):
    return int(beats) * SAMPLES_PER_BEAT

def seq_cycles_for(seconds):
    return max(1, int(round(seconds * SEQ_CLK_HZ)))

def pack_iq(i, q):
    if len(i) != len(q):
        raise ValueError('I/Q length mismatch')
    out = np.empty(2*len(i), dtype=np.int16)
    out[0::2] = i
    out[1::2] = q
    return out
print('Register map ready.')


Register map ready.


## Cell 5 — Waveform primitives

Four composable functions operating on plain complex128 arrays. A sine is just a chirp with
equal start/end frequency -- there is no separate "sine" case. `waveform(*parts)` takes any
number of arrays, in any order, including nested `waveform(...)` calls, and concatenates them.
Frequencies are always **absolute MHz**; each function subtracts the NCO internally so the
program you write never has to do that math by hand.


In [5]:
# ================= WAVEFORM PRIMITIVES =================
def chirp(f0_mhz, f1_mhz, duration_s, nco_mhz):
    """Complex128 I/Q array. f0_mhz==f1_mhz gives a plain sine. Frequencies are absolute;
    the NCO offset is subtracted internally."""
    n = samples_for(beats_for(duration_s))
    t = np.arange(n, dtype=np.float64) / FS_HZ
    f0 = (f0_mhz - nco_mhz) * 1e6
    f1 = (f1_mhz - nco_mhz) * 1e6
    k = (f1 - f0) / duration_s if duration_s > 0 else 0.0
    ph = 2*np.pi*(f0*t + 0.5*k*t*t)
    return np.exp(1j*ph).astype(np.complex128)

def zeros(duration_s):
    n = samples_for(beats_for(duration_s))
    return np.zeros(n, dtype=np.complex128)

def envelope(arr, kind="gaussian", **params):
    n = len(arr)
    if kind == "gaussian":
        sigma = params.get("sigma", 0.25) * n
        x = np.arange(n) - (n-1)/2.0
        win = np.exp(-0.5*(x/sigma)**2)
    else:
        raise ValueError(f"unknown envelope kind: {kind}")
    return arr * win

def waveform(*parts):
    """Concatenate any number of arrays (chirp/zeros/envelope/other waveform() results,
    in any order) into one flat array."""
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.complex128)

def quantize_iq(arr):
    """Final step only -- turns a complex128 array into the int16 I/Q byte stream the
    DMA actually sends. Do this once, on the final per-lane concatenated array."""
    i = np.round(AMPLITUDE * arr.real).astype(np.int16)
    q = np.round(AMPLITUDE * arr.imag).astype(np.int16)
    return pack_iq(i, q)

def chirp_train(freqs_mhz, pulse_s, gap_s, nco_mhz):
    """Convenience wrapper: a series of tones/chirps separated by zero gaps, combined into
    one waveform -- e.g. your 190-194 MHz probe train. Just waveform() calls under the hood."""
    parts = []
    for idx, f in enumerate(freqs_mhz):
        parts.append(chirp(f, f, pulse_s, nco_mhz))
        if idx < len(freqs_mhz) - 1:
            parts.append(zeros(gap_s))
    return waveform(*parts)

print('Waveform primitives ready.')


Waveform primitives ready.


## Cell 6 — Row helpers: `repeat()` and `print_program()`

`repeat()` is plain list repetition -- row dicts are only ever read at compile time, never
mutated, so sharing the same waveform array across repeated rows is safe (no copies made
until the final per-lane concatenation in Cell 8).

`print_program()` is the read-only view you asked for: row index is never something you type
into the program, it only appears here, at print time, via `enumerate`.


In [6]:
# ================= ROW HELPERS =================
def repeat(rows, n):
    """Plain list repetition -- the SAME row dicts (and the SAME waveform arrays inside them)
    are reused across every repetition, not copied. That's fine and cheap as long as rows are
    only ever read (which is all compile_program() below does), but it does mean you can't
    tweak "just the last repetition" by mutating a row in place -- build that one repetition
    as its own separate block instead (see Cell 7 for an example: the state-prep block's final
    repetition is written out separately so it can skip the refill flag the other 19 need)."""
    return list(rows) * n

def _row_lanes(row):
    lanes = []
    for lane in TX_LANES:
        if lane in row.get("tx", {}):
            lanes.append(lane)
    for lane in RX_LANES:
        if lane in row.get("rx", {}):
            lanes.append(lane)
    return lanes

def print_program(rows):
    """Read-only view of a compiled row list. Row index only ever appears here (via
    enumerate) -- it is never something you type into the program itself."""
    print(f"{'row':>4}  {'gap_s':>10}  {'lanes (dur_s)':<50} {'refill'}")
    for idx, row in enumerate(rows):
        parts = []
        for lane in _row_lanes(row):
            if lane in TX_LANES:
                arr = row['tx'][lane]
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms({len(arr)}smp)")
            else:
                dur_s = row['rx'][lane]
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms")
        refill = ','.join(row.get('refill', [])) or '-'
        print(f"{idx:>4}  {row['gap_s']*1e3:>9.4f}ms  {', '.join(parts):<50} {refill}")

print('repeat() and print_program() ready.')


repeat() and print_program() ready.


## Cell 7 — The actual program (edit this cell for your sequence)

State prep / spectroscopy / detection, written as plain rows. `gap_s` for each row is set to
just past that row's own longest active duration (matching the `+5us`-style margins already
used in the qualification notebook) so the next row fires right after this one finishes.

**Flagged for benchmarking:** the `refill` on the state-prep block's second row asks the
compiler to schedule a TX_B buffer refill inside TX_A's 2 ms window -- see Cell 8's warning
if `TRANSFER_CALL_OVERHEAD_S_ESTIMATE` doesn't comfortably fit that gap. Note the *final*
repetition is written as its own block without the `refill` flag -- there's no more TX_B data
coming after the last repetition, so flagging it there would ask the compiler to prepare a
chunk that's never used. This is also why the last repetition can't just be "the same block
with refill removed" via mutation -- see the `repeat()` note in Cell 6.


In [7]:
# ================= PROGRAM DEFINITION (edit me) =================
ROW_MARGIN_S = 5e-6   # small pad added to each row's own active duration when setting gap_s

def gap_after(*durations_s):
    return max(durations_s) + ROW_MARGIN_S

# ---- Phase 1: state prep (DAC A: 15MHz sine + 5-10MHz chirp, DAC B: 5-25MHz 20ms sweep) ----
state_prep_dacA = waveform(chirp(15, 15, 1e-3, DAC_A_NCO_MHZ), chirp(5, 10, 1e-3, DAC_A_NCO_MHZ))
state_prep_dacB = chirp(5, 25, 20e-3, DAC_B_NCO_MHZ)

state_prep_block = [
    {"gap_s": gap_after(2e-3), "tx": {"TX_A": state_prep_dacA}},
    {"gap_s": gap_after(20e-3), "tx": {"TX_B": state_prep_dacB}, "refill": ["TX_B"]},
]
# Final repetition: same content, but no 'refill' flag -- nothing needs to load after it.
state_prep_block_final = [
    state_prep_block[0],
    {"gap_s": gap_after(20e-3), "tx": {"TX_B": state_prep_dacB}},
]
STATE_PREP_REPS = 20
state_prep_rows = repeat(state_prep_block, STATE_PREP_REPS - 1) + state_prep_block_final

# ---- Phase 2: RF spectroscopy (both DACs silent; TX DMAs may still refill here) ----
SPECTROSCOPY_S = 5e-3
spectroscopy_rows = [
    {"gap_s": gap_after(SPECTROSCOPY_S), "tx": {}, "rx": {}},
]

# ---- Phase 3: detection (probe train on DAC A + simultaneous capture on both RX lanes) ----
probe = chirp_train([190, 191, 192, 193, 194], pulse_s=10e-6, gap_s=1e-6, nco_mhz=DAC_A_NCO_MHZ)
CAPTURE_S = len(probe) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ   # capture exactly spans the probe for now

detection_rows = [
    {"gap_s": gap_after(CAPTURE_S), "tx": {"TX_A": probe}, "rx": {"RX_A": CAPTURE_S, "RX_B": CAPTURE_S}},
]

rows = state_prep_rows + spectroscopy_rows + detection_rows
print_program(rows)


 row       gap_s  lanes (dur_s)                                      refill
   0     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   1    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   2     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   3    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   4     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   5    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   6     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   7    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   8     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   9    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
  10     2.0050ms  TX_A:2.0000ms(245760smp)                           -
  11    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
  12     2.0050ms  TX_A:2.0000ms(245760smp

## Cell 8 — Compile: derive mask/dur, concatenate per-lane TX buffers, resolve refill chunks

`mask` and `DUR0..DUR3` are never typed by hand -- they are derived here, per row, from
whichever `tx`/`rx` keys that row actually has. Each TX lane's full waveform is built by
concatenating, in row order, every array that lane contributes -- there is exactly one array
per lane in the whole design, so a refill chunk is always "the next already-written slice of
that one array," never something recomputed separately.


In [8]:
# ================= COMPILE PROGRAM =================
def compile_program(rows):
    table = []  # list of dicts: {mask, gap_cycles, dur: {lane: beats}}
    tx_chunks = {lane: [] for lane in TX_LANES}         # arrays contributed by each row, in order
    tx_chunk_bounds = {lane: [0] for lane in TX_LANES}  # sample offsets where a new chunk starts
    rx_total_s = {lane: 0.0 for lane in RX_LANES}
    refill_points = []  # (row_idx, lane) pairs, checked against gap_s below

    for idx, row in enumerate(rows):
        mask = 0
        dur = {}
        for lane in TX_LANES:
            arr = row.get("tx", {}).get(lane)
            if arr is not None:
                mask |= (1 << LANE_BIT[lane])
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                dur[lane] = beats_for(dur_s)
                tx_chunks[lane].append(arr)
        for lane in RX_LANES:
            dur_s = row.get("rx", {}).get(lane)
            if dur_s:
                mask |= (1 << LANE_BIT[lane])
                dur[lane] = beats_for(dur_s)
                rx_total_s[lane] += dur_s
        for lane in row.get("refill", []):
            refill_points.append((idx, lane))
            tx_chunk_bounds[lane].append(sum(len(a) for a in tx_chunks[lane]))
        table.append({"mask": mask, "gap_cycles": seq_cycles_for(row["gap_s"]), "dur": dur})

    tx_full = {lane: (np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.complex128))
               for lane, chunks in tx_chunks.items()}
    for lane in TX_LANES:
        tx_chunk_bounds[lane].append(len(tx_full[lane]))

    return table, tx_full, tx_chunk_bounds, refill_points, rx_total_s

table, tx_full, tx_chunk_bounds, refill_points, rx_total_s = compile_program(rows)

if len(table) > 256:
    raise ValueError(f"Program has {len(table)} rows, exceeds the 256-row Pulse_Sequencer build.")

# ---- Consistency + capacity checks (fail before touching hardware) ----
for lane in TX_LANES:
    print(f"{lane}: {len(tx_full[lane]):,} samples total across the whole program")

for lane, secs in rx_total_s.items():
    if secs > CAP_MAX_S:
        raise RuntimeError(f"{lane}: total capture {secs*1e3:.1f} ms exceeds the {CAP_MAX_S*1e3:.0f} ms RX DMA limit.")
    print(f"{lane}: {secs*1e3:.3f} ms total capture this program")

# ---- Refill feasibility warning (needs TRANSFER_CALL_OVERHEAD_S_ESTIMATE benchmarked for real) ----
for row_idx, lane in refill_points:
    gap_s = rows[row_idx]["gap_s"]
    if gap_s < TRANSFER_CALL_OVERHEAD_S_ESTIMATE:
        print(f"WARNING: refill of {lane} scheduled in row {row_idx}'s gap ({gap_s*1e3:.3f} ms) "
              f"is tighter than the estimated transfer()-call overhead "
              f"({TRANSFER_CALL_OVERHEAD_S_ESTIMATE*1e3:.3f} ms). Benchmark before trusting this.")

# ---- Per-lane chunk size vs DMA transfer limit ----
for lane in TX_LANES:
    bounds = tx_chunk_bounds[lane]
    for c0, c1 in zip(bounds[:-1], bounds[1:]):
        chunk_bytes = (c1 - c0) * 4
        if chunk_bytes > DMA_MAX_BYTES:
            raise RuntimeError(f"{lane}: a chunk between samples {c0}-{c1} is {chunk_bytes:,} bytes, "
                                f"exceeds the {DMA_MAX_BYTES:,}-byte single-DMA-transfer limit. "
                                f"Add a 'refill' flag on an earlier row to split it further.")
    chunk_sizes = [(b1-b0)*4 for b0, b1 in zip(bounds[:-1], bounds[1:])]
    print(f"{lane}: {len(bounds)-1} chunk(s), sizes(bytes)={chunk_sizes}")

print("PASS: program compiled and passes capacity/consistency checks.")


TX_A: 4,921,840 samples total across the whole program
TX_B: 49,152,000 samples total across the whole program
RX_A: 0.054 ms total capture this program
RX_B: 0.054 ms total capture this program
TX_A: 1 chunk(s), sizes(bytes)=[19687360]
TX_B: 20 chunk(s), sizes(bytes)=[9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400]
PASS: program compiled and passes capacity/consistency checks.


## Cell 9 — RFDC NCO setup

Sets both DAC mixers once, before the run, matching the qualification notebook's explicit
`MixerSettings`/`UpdateEvent` pattern.


In [9]:
# ================= RFDC NCO SETUP =================
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    ms['Freq'] = float(freq)
    ms['PhaseOffset'] = 0.0
    ms['EventSource'] = 2
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)

for dac, label, freq in ((dac_b, 'DAC B', DAC_B_NCO_MHZ), (dac_a, 'DAC A', DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    assert abs(float(ms['Freq']) - freq) < 1e-6
print('PASS: DAC mixer state matches config.')

def set_nco_during_gap(dac, freq_mhz, gap_deadline_perf_counter):
    """Only safe to call while that DAC's TX_Multi_Gate lane is CLOSED (output forced to
    zero) -- e.g. during a row where that lane has no 'tx' entry. AXI-Lite mixer writes are
    microsecond-scale, so this comfortably fits inside a >=1ms gap, but the caller must ensure
    the current time is actually within such a gap."""
    ms = dac.MixerSettings
    ms['Freq'] = float(freq_mhz)
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)
    if time.perf_counter() > gap_deadline_perf_counter:
        print(f"WARNING: NCO write to {freq_mhz} MHz may have missed its gap deadline.")


PASS: DAC mixer state matches config.


## Cell 10 — Commit the compiled table to the sequencer

Same register-write sequence as `program_production_sequence()` in the qualification
notebook, just driven by the compiled `table` instead of a hand-written row list.


In [10]:
# ================= COMMIT TABLE TO HARDWARE =================
def commit_table(table):
    seq.mmio.write(SEQ_TTL_MODE, 0)
    seq.mmio.write(SEQ_ENABLE, 0)
    seq.mmio.write(SEQ_RESET, 1)
    time.sleep(100e-6)

    for idx, row in enumerate(table):
        seq.mmio.write(SEQ_ROW_SEL, idx)
        seq.mmio.write(SEQ_MASK, row["mask"])
        seq.mmio.write(SEQ_GAP, row["gap_cycles"])
        seq.mmio.write(SEQ_DUR0, row["dur"].get("TX_B", 0))
        seq.mmio.write(SEQ_DUR1, row["dur"].get("TX_A", 0))
        seq.mmio.write(SEQ_DUR2, row["dur"].get("RX_B", 0))
        seq.mmio.write(SEQ_DUR3, row["dur"].get("RX_A", 0))
        seq.mmio.write(SEQ_COMMIT, 1)

    seq.mmio.write(SEQ_TABLE_LEN, len(table))
    print(f"PASS: committed {len(table)} rows to the sequencer.")

commit_table(table)


PASS: committed 42 rows to the sequencer.


## Cell 11 — Allocate DMA buffers and load the first TX chunk

One PYNQ buffer per TX chunk (per lane), one RX buffer per RX lane sized to that lane's total
capture length this program. Quantization to int16 happens exactly once here, on the final
concatenated waveform, per chunk.


In [11]:
# ================= DMA BUFFERS =================
tx_buffers = {lane: [] for lane in TX_LANES}   # list of allocate()'d buffers, in chunk order
for lane in TX_LANES:
    bounds = tx_chunk_bounds[lane]
    for c0, c1 in zip(bounds[:-1], bounds[1:]):
        chunk = quantize_iq(tx_full[lane][c0:c1])
        buf = base.device.get_memory_by_idx(1).allocate(shape=chunk.shape, dtype=np.int16)
        buf[:] = chunk
        buf.flush()
        tx_buffers[lane].append(buf)
    sizes = [b.nbytes for b in tx_buffers[lane]]
    print(f"{lane}: {len(tx_buffers[lane])} chunk buffer(s) allocated, sizes(bytes)={sizes}")

# RX is double-buffered: 2 physical buffers per lane, alternated by shot parity. This is what
# actually lets the Ethernet send of shot k's data run in the background while shot k+1's
# hardware (prepare_shot/arm_shot, TX-only rows) is already underway -- see Cell 12's
# background sender and Cell 14's run_shot(). A single buffer would force every shot to wait
# for the previous shot's send to fully finish before the DMA could reuse it.
rx_buffers = {}   # lane -> [slot0_buf, slot1_buf]
for lane in RX_LANES:
    capture_beats = beats_for(rx_total_s[lane])
    capture_samples = samples_for(capture_beats)
    rx_buffers[lane] = [allocate(shape=(capture_samples,), dtype=np.int16) for _ in range(2)]
    print(f"{lane}: RX buffer {rx_buffers[lane][0].nbytes:,} bytes x2 (double-buffered)")

print('PASS: DMA buffers allocated and first TX chunks loaded.')


TX_A: 1 chunk buffer(s) allocated, sizes(bytes)=[19687360]
TX_B: 20 chunk buffer(s) allocated, sizes(bytes)=[9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400, 9830400]
RX_A: RX buffer 13,280 bytes x2 (double-buffered)
RX_B: RX buffer 13,280 bytes x2 (double-buffered)
PASS: DMA buffers allocated and first TX chunks loaded.


## Cell 12 -- DMA/loop helpers, `dump_state()`, Ethernet protocol, background sender

DMA/gate wait helpers are unchanged from the qualification notebook. Two additions:

- **`dump_state()`** -- prints ROW_PTR/TTL_STATUS/CDC_OVERRUN plus every gate's and DMA's
  status register in one place. Called automatically whenever the run loop hits an exception
  (Cell 14), and safe to call manually from any cell below if something looks wrong.
- **Background Ethernet sender** -- a dedicated thread that owns all `SHOT` socket writes, fed
  by a queue. This is what makes the Ethernet overlap in Cell 13 real instead of aspirational:
  the run loop hands off a shot's data and continues immediately to the next shot's hardware,
  rather than blocking on `sock.sendall()`. See Cell 14 for how this pairs with the
  double-buffered RX allocation from Cell 11.


In [12]:
# ================= DMA / LOOP HELPERS =================
def dma_status(ch): return int(ch._mmio.read(int(ch._offset)+0x04))
def dma_flags(v):
    out = ['halted' if v&1 else 'running']
    if v&2: out.append('idle')
    if v&0x10: out.append('INTERNAL_ERR')
    if v&0x20: out.append('SLAVE_ERR')
    if v&0x40: out.append('DECODE_ERR')
    return '|'.join(out)

def ensure_running(ch, label):
    st = dma_status(ch)
    if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error before shot: 0x{st:08x} ({dma_flags(st)})')
    if st & DMASR_HALTED:
        ch.start(); t0 = time.perf_counter()
        while time.perf_counter()-t0 < 0.2:
            st = dma_status(ch)
            if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error restarting: 0x{st:08x} ({dma_flags(st)})')
            if not (st & DMASR_HALTED): return
            time.sleep(1e-5)
        raise TimeoutError(f'{label}: DMA did not become running: 0x{st:08x}')

def wait_dma_idle(ch, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = dma_status(ch)
        if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error: 0x{st:08x} ({dma_flags(st)})')
        if st & DMASR_IDLE: return
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: DMA not idle: 0x{st:08x} ({dma_flags(st)})')
        time.sleep(50e-6)

def wait_cap_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(CAP_STATUS))
        if not (st & CAP_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: Capture_Gate busy: 0x{st:08x}')
        time.sleep(100e-6)

def wait_tx_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(TXM_STATUS))
        if not (st & TX_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: TX gate busy: 0x{st:08x}')
        time.sleep(20e-6)

def discard_rx(buf):
    inv = getattr(buf, 'invalidate', None)
    if inv:
        try: inv()
        except Exception: pass

def ttl_arm(s): s.mmio.write(SEQ_TTL_ARM, 1)
def ttl_status_val(s): return int(s.mmio.read(SEQ_TTL_STATUS))
def ttl_edge_count(s): return int(s.mmio.read(SEQ_TTL_EDGE_COUNT))

def dump_state(label=""):
    """Prints every core's status registers in one place -- called automatically on any
    exception in the run loop, and safe to call manually any time things look wrong instead
    of hunting through individual mmio.read() calls by hand."""
    print(f"---- dump_state: {label} ----")
    try:
        print(f"  seq:   ROW_PTR={int(seq.mmio.read(SEQ_ROW_PTR))} "
              f"TTL_STATUS=0x{int(seq.mmio.read(SEQ_TTL_STATUS)):x} "
              f"CDC_OVERRUN=0x{int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF:x} "
              f"EDGE_COUNT={int(seq.mmio.read(SEQ_TTL_EDGE_COUNT))}")
    except Exception as e:
        print(f"  seq: <read failed: {e}>")
    for name, g in [("tx_a", tx_a), ("tx_b", tx_b)]:
        try:
            st = int(g.mmio.read(TXM_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&TX_BUSY)}, overrun={bool(st&TX_OVERRUN)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, g in [("cap_a", cap_a), ("cap_b", cap_b)]:
        try:
            st = int(g.mmio.read(CAP_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&CAP_BUSY)}, overflow={bool(st&CAP_OVERFLOW)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, ch in [("RX-A", dma_ra.recvchannel), ("RX-B", dma_rb.recvchannel),
                      ("TX-A", dma_ta.sendchannel), ("TX-B", dma_tb.sendchannel)]:
        try:
            st = dma_status(ch)
            print(f"  {name}: 0x{st:08x} ({dma_flags(st)}) transferred={int(ch.transferred)}")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    print("-" * (18 + len(label)))

# ================= ETHERNET PROTOCOL (extends the qualification notebook's protocol) =================
MAGIC_HELLO = b'RFDC'
MAGIC_SHOT  = b'SHOT'
MAGIC_DONE  = b'DONE'   # sent right after RUN_DONE, before the data transfer
ACK = b'OK'

def _recv_exact(sock, n):
    buf = bytearray(n); view = memoryview(buf); got = 0
    while got < n:
        r = sock.recv_into(view[got:], n - got)
        if r == 0: raise ConnectionError('PC closed the connection mid-transfer')
        got += r
    return bytes(buf)

def eth_connect(host, port, timeout_s):
    s = socket.create_connection((host, port), timeout=timeout_s)
    s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    return s

def _send_json(sock, magic, obj):
    payload = json.dumps(obj).encode('utf-8')
    sock.sendall(magic + struct.pack('>I', len(payload)) + payload)

def eth_handshake(sock, meta):
    _send_json(sock, MAGIC_HELLO, meta)
    ack = _recv_exact(sock, 2)
    if ack != ACK: raise RuntimeError(f'PC receiver did not ACK handshake, got {ack!r}')

def eth_send_done(sock, shot_num):
    """Tells a PC-side AWG-control notebook this run's hardware is complete -- sent before
    the (slower) data transfer so the next external trigger doesn't wait on it."""
    _send_json(sock, MAGIC_DONE, {'shot': shot_num})
    ack = _recv_exact(sock, 2)
    if ack != ACK: raise RuntimeError(f'shot {shot_num}: PC did not ACK DONE, got {ack!r}')

def eth_send_shot(sock, shot_num, header, rx_arrays):
    hdr = dict(header); hdr['shot'] = shot_num
    _send_json(sock, MAGIC_SHOT, hdr)
    for arr in rx_arrays:
        sock.sendall(memoryview(np.asarray(arr)))
    ack = _recv_exact(sock, 2)
    if ack != ACK: raise RuntimeError(f'shot {shot_num}: PC receiver did not ACK, got {ack!r}')

# ================= BACKGROUND ETHERNET SENDER =================
# A single dedicated thread owns every SHOT-data socket write, in shot order (a raw TCP
# socket can't have two threads calling sendall() on it concurrently -- that would interleave
# and corrupt the stream, so this is the one thread allowed to touch it for SHOT payloads).
# The run loop enqueues a shot's data right after that shot's DMA is confirmed complete and
# valid, then moves on immediately to the next shot's hardware -- prepare_shot()/arm_shot()
# and any TX-only rows run concurrently with this thread's send. That's the actual Ethernet
# overlap; the earlier single-buffered version instead blocked on the full transfer before
# it could arm the next shot, which didn't overlap anything despite the intent.
_eth_queue = queue.Queue()
_eth_stop = object()          # sentinel: tells the sender thread to exit
_eth_error = []                # populated if a send raises; checked every run_shot() call
_eth_send_times = []           # (shot_num, elapsed_s) for each completed send, for the summary

def _eth_sender_loop():
    while True:
        item = _eth_queue.get()
        if item is _eth_stop:
            return
        shot_num, header, arrays, done_event = item
        t0 = time.perf_counter()
        try:
            eth_send_shot(eth_sock, shot_num, header, arrays)
            _eth_send_times.append((shot_num, time.perf_counter() - t0))
        except Exception as e:
            _eth_error.append((shot_num, e))
        finally:
            done_event.set()   # signal: this RX buffer slot is safe for the DMA to reuse

print('DMA/loop helpers, dump_state(), and background Ethernet sender ready.')


DMA/loop helpers, dump_state(), and background Ethernet sender ready.


## Debug -- dump current hardware state

Run this cell any time (before, during between-cell debugging, or after a failed run) to
see every core's status registers at once. `dump_state()` itself is defined in Cell 12
(needs the register map from Cell 4 and the IP handles from Cell 3), so this cell only
works after those have run -- it's placed here as a quick-access marker; feel free to
re-run it from anywhere below Cell 12 too.


In [13]:
dump_state("manual check")


---- dump_state: manual check ----
  seq:   ROW_PTR=0 TTL_STATUS=0x0 CDC_OVERRUN=0x0 EDGE_COUNT=0
  tx_a: STATUS=0x0 (busy=False, overrun=False)
  tx_b: STATUS=0x0 (busy=False, overrun=False)
  cap_a: STATUS=0x0 (busy=False, overflow=False)
  cap_b: STATUS=0x0 (busy=False, overflow=False)
  RX-A: 0x00000000 (running) transferred=0
  RX-B: 0x00000000 (running) transferred=0
  TX-A: 0x00000000 (running) transferred=0
  TX-B: 0x00000000 (running) transferred=0
------------------------------


## Cell 13 -- Ethernet transfer time estimate (informational)

This is now a **diagnostic estimate only**, not something the run loop sleeps on. With the
double-buffered RX + background sender (Cell 12) and `run_shot()` (Cell 14), a shot's send
overlaps with literally the next shot's entire hardware run, not just the fraction of one row
that used to be computed here -- so as long as one Ethernet transfer finishes before that same
buffer slot is needed *two* shots later, there's no wait at all. That's normally a lot more
slack than the single-row estimate below suggests.

This cell still prints the estimate because it's useful for spotting a real problem *before*
you run anything: if the estimated transfer time is close to or longer than a full shot's
total duration, double buffering with only 2 slots won't be enough headroom, and you'd see
real (measured, not estimated) buffer-reuse waits printed live during the run -- see
`buffer_wait_s` in Cell 14/16.


In [14]:
# ================= ETHERNET TRANSFER TIME ESTIMATE (informational) =================
def estimate_eth_transfer_s(capture_bytes_per_lane, n_lanes=2, mbps=None):
    mbps = mbps if mbps is not None else ETHERNET_LINK_MBPS_MEASURED
    total_bits = capture_bytes_per_lane * n_lanes * 8
    return total_bits / (mbps * 1e6) + ETH_MARGIN_S

def total_row_time_s(table):
    """Wall-clock duration of one full pass through the program (sum of every row's gap_s).
    Used only as a sanity check against the estimated Ethernet time below -- the actual
    overlap is measured live during the run, not scheduled from this number."""
    return sum(row["gap_cycles"] for row in table) / SEQ_CLK_HZ

_capture_bytes = int(beats_for(max(rx_total_s.values())) * SAMPLES_PER_BEAT * 2)
_eth_est_s = estimate_eth_transfer_s(_capture_bytes)
_shot_period_s = total_row_time_s(table)

print(f"Estimated Ethernet transfer time per shot: {_eth_est_s*1e3:.1f} ms "
      f"(at {ETHERNET_LINK_MBPS_MEASURED} Mbps measured, +{ETH_MARGIN_S*1e3:.1f} ms margin)")
print(f"One full program pass (shot period): {_shot_period_s*1e3:.1f} ms")
if _eth_est_s > _shot_period_s:
    print(f"WARNING: estimated Ethernet time exceeds one shot period -- double buffering "
          f"(2 slots) will not fully hide the transfer; expect real buffer_wait_s > 0 during "
          f"the run. Consider a lower LOOP rate, a 3rd buffer slot, or fixing link speed first.")
else:
    print("PASS: estimated Ethernet time comfortably fits within one shot period; double "
          "buffering should fully hide it (buffer_wait_s should read ~0 during the run).")


Estimated Ethernet transfer time per shot: 6.1 ms (at 190 Mbps measured, +5.0 ms margin)
One full program pass (shot period): 445.3 ms
PASS: estimated Ethernet time comfortably fits within one shot period; double buffering should fully hide it (buffer_wait_s should read ~0 during the run).


## Cell 14 -- Run loop

`prepare_shot`/`arm_shot` now take a buffer `slot` (0 or 1, alternating by shot parity) and,
before letting the DMA write into that slot, wait on that slot's previous occupant's
background-send completion `Event` if it hasn't fired yet (this is the only place a wait can
happen -- if the background sender easily keeps up, this wait is ~0 every time).

Additions vs. the earlier version: (1) mid-run refills are unchanged (poll `SEQ_ROW_PTR`,
fire the next chunk's `transfer()` once that lane's DMA reports idle); (2) `MAGIC_DONE` is
still sent synchronously right after `RUN_DONE`, before anything RX-related, exactly as
before; (3) the RX payload itself is now handed to the background sender instead of sent
inline, so `run_shot()` returns as soon as the hand-off is queued -- it does not wait for the
transfer to finish; (4) a capture-gate overflow still tags the shot rather than raising; (5)
`dump_state()` is called automatically on any exception so a failure is debuggable without
re-running.


In [15]:
# ================= RUN LOOP =================
slot_events = {0: None, 1: None}   # last-in-flight send's completion Event per buffer slot

def prepare_shot(slot):
    wait_cap_idle(cap_a, "RX_A", 2.0); wait_cap_idle(cap_b, "RX_B", 2.0)
    wait_tx_idle(tx_a, "TX_A", 1.0); wait_tx_idle(tx_b, "TX_B", 1.0)

    ev = slot_events[slot]
    if ev is not None and not ev.is_set():
        t_wait0 = time.perf_counter()
        if not ev.wait(timeout=ETH_TIMEOUT_S):
            raise TimeoutError(f"slot {slot}: previous shot's Ethernet send did not finish "
                                f"within {ETH_TIMEOUT_S}s -- background sender may be stuck.")
        wait_s = time.perf_counter() - t_wait0
        prepare_shot.last_buffer_wait_s = wait_s
        if DEBUG_VERBOSE and wait_s > 1e-3:
            print(f"  (waited {wait_s*1e3:.1f} ms for slot {slot}'s previous Ethernet send)")
    else:
        prepare_shot.last_buffer_wait_s = 0.0

    cap_a.mmio.write(CAP_CLEAR, CAP_OVERFLOW); cap_b.mmio.write(CAP_CLEAR, CAP_OVERFLOW)
    cap_a.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_A"]))
    cap_b.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_B"]))
    tx_a.mmio.write(TXM_STATUS, TX_OVERRUN); tx_b.mmio.write(TXM_STATUS, TX_OVERRUN)
    seq.mmio.write(SEQ_ENABLE, 0); seq.mmio.write(SEQ_RESET, 1); time.sleep(100e-6)
    seq.mmio.write(SEQ_CDC_OVERRUN, 0xF); seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1)
    seq.mmio.write(SEQ_TTL_MODE, 1)
    for ch, label in [(dma_ra.recvchannel,'RX-A'), (dma_rb.recvchannel,'RX-B'),
                       (dma_ta.sendchannel,'TX-A'), (dma_tb.sendchannel,'TX-B')]:
        ensure_running(ch, label)
prepare_shot.last_buffer_wait_s = 0.0

def arm_shot(slot):
    dma_ra.recvchannel.transfer(rx_buffers["RX_A"][slot])
    dma_rb.recvchannel.transfer(rx_buffers["RX_B"][slot])
    dma_ta.sendchannel.transfer(tx_buffers["TX_A"][0])
    dma_tb.sendchannel.transfer(tx_buffers["TX_B"][0])
    time.sleep(5e-3)
    for ch, label in [(dma_ra.recvchannel,'RX-A'), (dma_rb.recvchannel,'RX-B'),
                       (dma_ta.sendchannel,'TX-A'), (dma_tb.sendchannel,'TX-B')]:
        st = dma_status(ch)
        if st & (DMASR_ERR_MASK | DMASR_HALTED):
            raise RuntimeError(f'{label}: bad after arm 0x{st:08x} ({dma_flags(st)})')

def _pending_refills():
    """List of [row_idx, lane, next_chunk_index] for this shot. Chunk 0 of every lane was
    already transferred in arm_shot(), so counting starts at 1."""
    counts = {lane: 1 for lane in TX_LANES}
    pending = []
    for row_idx, lane in refill_points:
        pending.append([row_idx, lane, counts[lane]])
        counts[lane] += 1
    return pending

def _try_refill(row_idx, lane, chunk_idx, k):
    """Returns True once this refill has been issued (or skipped as out-of-range). Only
    issues the transfer once the channel actually reports idle (the previous chunk is still
    draining otherwise), returning False so the caller re-polls on its next iteration."""
    if chunk_idx >= len(tx_buffers[lane]):
        return True  # nothing more to load for this lane
    ch = DMA_LANE_TO_CHANNEL[lane]
    st = dma_status(ch)
    if st & DMASR_ERR_MASK:
        raise RuntimeError(f'shot {k}: {lane} DMA error before refill (row {row_idx}): 0x{st:08x} ({dma_flags(st)})')
    if not (st & DMASR_IDLE):
        return False  # previous chunk still draining; try again next poll
    ch.transfer(tx_buffers[lane][chunk_idx])
    return True

DMA_LANE_TO_CHANNEL = {"TX_A": dma_ta.sendchannel, "TX_B": dma_tb.sendchannel}

last_shot_end_t = None

def run_shot(k):
    global last_shot_end_t
    if _eth_error:
        shot_num, exc = _eth_error[0]
        raise RuntimeError(f"background Ethernet sender failed on shot {shot_num}: {exc!r} "
                            f"(all sends after this are suspect -- fix and restart the run)")

    slot = (k - 1) % 2
    t0 = time.perf_counter()
    try:
        prepare_shot(slot); arm_shot(slot)
    except Exception:
        dump_state(f"shot {k} prepare/arm failure")
        raise
    t_armed = time.perf_counter()
    gap_before_s = (t_armed - last_shot_end_t) if last_shot_end_t is not None else None

    pending = _pending_refills()
    ttl_arm(seq)

    deadline = time.perf_counter() + 5.0   # generous; not the per-shot timing signal, see summary cell
    while True:
        row_ptr = int(seq.mmio.read(SEQ_ROW_PTR))
        while pending and pending[0][0] <= row_ptr:
            row_idx, lane, chunk_idx = pending[0]
            if _try_refill(row_idx, lane, chunk_idx, k):
                if DEBUG_VERBOSE: print(f"  shot {k}: refilled {lane} chunk {chunk_idx} at row {row_idx}")
                pending.pop(0)
            else:
                break  # this lane's DMA isn't idle yet -- retry on the next poll, don't skip ahead

        cdc = int(seq.mmio.read(SEQ_CDC_OVERRUN) & 0xF)
        if cdc:
            dump_state(f"shot {k} CDC overrun")
            raise RuntimeError(f'shot {k}: CDC overrun 0x{cdc:x}')
        if int(seq.mmio.read(SEQ_TTL_STATUS)) & TTL_RUN_DONE: break
        if time.perf_counter() > deadline:
            dump_state(f"shot {k} RUN_DONE timeout")
            raise TimeoutError(f'shot {k}: RUN_DONE timeout, row_ptr={row_ptr}')
        time.sleep(0.5e-3)

    seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1)
    row = int(seq.mmio.read(SEQ_ROW_PTR))

    eth_send_done(eth_sock, k)   # still synchronous -- tiny message, sent before the RX wait below

    tout = max(2.0, max(rx_total_s.values()) + 1.0)
    try:
        wait_cap_idle(cap_a, "RX_A", tout); wait_cap_idle(cap_b, "RX_B", tout)
        wait_dma_idle(dma_ra.recvchannel, 'RX-A', tout); wait_dma_idle(dma_rb.recvchannel, 'RX-B', tout)
        wait_dma_idle(dma_ta.sendchannel, 'TX-A', 2.0); wait_dma_idle(dma_tb.sendchannel, 'TX-B', 2.0)
    except Exception:
        dump_state(f"shot {k} post-run wait failure")
        raise
    dma_ra.recvchannel.wait(); dma_rb.recvchannel.wait(); dma_ta.sendchannel.wait(); dma_tb.sendchannel.wait()

    ca = int(cap_a.mmio.read(CAP_STATUS)); cb = int(cap_b.mmio.read(CAP_STATUS))
    overflow = bool(ca & CAP_OVERFLOW) or bool(cb & CAP_OVERFLOW)   # tagged, not raised
    rx_bytes_transferred = {'RX_A': int(dma_ra.recvchannel.transferred),
                             'RX_B': int(dma_rb.recvchannel.transferred)}

    discard_rx(rx_buffers["RX_A"][slot]); discard_rx(rx_buffers["RX_B"][slot])
    shot_end_t = time.perf_counter(); last_shot_end_t = shot_end_t

    # Hand off to the background sender and return immediately -- THIS is the actual overlap:
    # the next call to run_shot() (prepare_shot/arm_shot and any TX-only rows) proceeds
    # concurrently with this shot's data still being sent to the PC.
    done_event = threading.Event()
    slot_events[slot] = done_event
    header = {'row': row, 'overflow': overflow, 'rx_bytes': rx_bytes_transferred}
    _eth_queue.put((k, header, [rx_buffers["RX_A"][slot], rx_buffers["RX_B"][slot]], done_event))

    return {'shot': k, 'row': row, 'overflow': overflow,
            'elapsed_s': time.perf_counter()-t0, 'gap_before_s': gap_before_s,
            'buffer_wait_s': prepare_shot.last_buffer_wait_s}

print('Run-loop functions ready.')


Run-loop functions ready.


## Cell 15 -- Connect to PC, handshake, start background sender, run the loop

Same handshake pattern as before. One addition: the background Ethernet sender thread
(Cell 12) is started here, right after the handshake ACK, so it's ready before the first
shot's data is queued.


In [16]:
# ================= CONNECT + RUN =================
LOOP_COUNT = 100
FAIL_FAST = True

print(f'Connecting to PC receiver at {PC_HOST}:{PC_PORT} ...')
eth_sock = eth_connect(PC_HOST, PC_PORT, ETH_TIMEOUT_S)
# Explicit per-lane sizes -- the PC receiver gets the ground truth rather than re-deriving
# sample/byte counts itself. rx_buffers[lane] is now [slot0, slot1] -- both slots are the
# same size, so index [0] for the size metadata.
rx_capture_samples = {lane: len(rx_buffers[lane][0]) for lane in RX_LANES}
rx_capture_bytes = {lane: int(rx_buffers[lane][0].nbytes) for lane in RX_LANES}
handshake_meta = dict(
    AXIS_BEAT_HZ=AXIS_BEAT_HZ, SAMPLES_PER_BEAT=SAMPLES_PER_BEAT, SEQ_CLK_HZ=SEQ_CLK_HZ, fs_hz=FS_HZ,
    DAC_A_NCO_MHZ=DAC_A_NCO_MHZ, DAC_B_NCO_MHZ=DAC_B_NCO_MHZ,
    rx_lanes=list(RX_LANES), rx_total_s=rx_total_s,
    rx_capture_samples=rx_capture_samples, rx_capture_bytes=rx_capture_bytes,
    LOOP_COUNT=LOOP_COUNT,
)
eth_handshake(eth_sock, handshake_meta)
print('PASS: PC receiver ACKed handshake.')

_eth_thread = threading.Thread(target=_eth_sender_loop, daemon=True, name='eth-sender')
_eth_thread.start()
print('Background Ethernet sender thread started.')

results = []
for k in range(1, LOOP_COUNT+1):
    try:
        r = run_shot(k); results.append(r)
        bw = r['buffer_wait_s']
        gap_str = f"{r['gap_before_s']*1e3:.1f}ms" if r['gap_before_s'] is not None else 'n/a'
        bw_str = f" buffer_wait={bw*1e3:.1f}ms" if bw > 1e-3 else ""
        print(f"shot {k:04d}: row={r['row']} overflow={r['overflow']} "
              f"gap_before={gap_str} elapsed={r['elapsed_s']:.3f}s{bw_str}")
    except Exception:
        print(f"shot {k} FAILED -- dumping hardware state:")
        dump_state(f"shot {k} exception")
        try: seq.mmio.write(SEQ_ENABLE, 0)
        except Exception: pass
        if FAIL_FAST: raise

print(f'Completed {len(results)}/{LOOP_COUNT} shots.')


Connecting to PC receiver at 192.168.3.136:5001 ...
PASS: PC receiver ACKed handshake.
Background Ethernet sender thread started.
  shot 1: refilled TX_B chunk 1 at row 1
  shot 1: refilled TX_B chunk 2 at row 3
  shot 1: refilled TX_B chunk 3 at row 5
  shot 1: refilled TX_B chunk 4 at row 7
  shot 1: refilled TX_B chunk 5 at row 9
  shot 1: refilled TX_B chunk 6 at row 11
  shot 1: refilled TX_B chunk 7 at row 13
  shot 1: refilled TX_B chunk 8 at row 15
  shot 1: refilled TX_B chunk 9 at row 17
  shot 1: refilled TX_B chunk 10 at row 19
  shot 1: refilled TX_B chunk 11 at row 21
  shot 1: refilled TX_B chunk 12 at row 23
  shot 1: refilled TX_B chunk 13 at row 25
  shot 1: refilled TX_B chunk 14 at row 27
  shot 1: refilled TX_B chunk 15 at row 29
  shot 1: refilled TX_B chunk 16 at row 31
  shot 1: refilled TX_B chunk 17 at row 33
  shot 1: refilled TX_B chunk 18 at row 35
  shot 1: refilled TX_B chunk 19 at row 37
shot 0001: row=0 overflow=False gap_before=n/a elapsed=0.467s
  sho

TimeoutError: TX-B: DMA not idle: 0x00001000 (running)

## Cell 16 -- Summary + cleanup

Drains the background sender (waits for any still-in-flight sends, then stops the thread)
before reporting stats, so the Ethernet timing numbers below reflect every shot, not just
whichever ones happened to finish sending before this cell ran.


In [ ]:
# ================= SUMMARY + CLEANUP =================
print('Draining background sender (waiting for any in-flight sends to finish)...')
_eth_queue.put(_eth_stop)
_eth_thread.join(timeout=ETH_TIMEOUT_S + 5.0)
if _eth_thread.is_alive():
    print("WARNING: background sender thread did not stop cleanly -- it may be stuck on a "
          "send. Check the network connection to the PC.")

if _eth_error:
    print(f"Background sender hit {len(_eth_error)} error(s):")
    for shot_num, exc in _eth_error:
        print(f"  shot {shot_num}: {exc!r}")

gaps = [r['gap_before_s'] for r in results if r['gap_before_s'] is not None]
buffer_waits = [r['buffer_wait_s'] for r in results]
send_times = [t for _, t in _eth_send_times]
overflows = sum(1 for r in results if r['overflow'])

if gaps:
    print(f"Max inter-shot dead time: {max(gaps)*1e3:.1f} ms")
    print(f"Mean inter-shot dead time: {sum(gaps)/len(gaps)*1e3:.1f} ms")
if send_times:
    print(f"Max Ethernet send time (actual, background thread): {max(send_times)*1e3:.1f} ms")
    print(f"Mean Ethernet send time: {sum(send_times)/len(send_times)*1e3:.1f} ms")
    print(f"Shots with a completed send: {len(send_times)}/{len(results)}")
if any(w > 1e-3 for w in buffer_waits):
    print(f"Max buffer-reuse wait (time a shot blocked on the previous send finishing): "
          f"{max(buffer_waits)*1e3:.1f} ms")
    print(f"Shots that had to wait at all: {sum(1 for w in buffer_waits if w > 1e-3)}/{len(results)}")
    print("(Nonzero here means the background sender couldn't keep up with 2 buffer slots -- "
          "see Cell 13's estimate/warning.)")
else:
    print("PASS: no shot ever waited on a buffer slot -- double buffering fully hid the "
          "Ethernet transfer time.")
print(f"Shots with capture overflow (dropped, per design): {overflows}/{len(results)}")

seq.mmio.write(SEQ_TTL_MODE, 0); seq.mmio.write(SEQ_ENABLE, 0)
try:
    eth_sock.shutdown(socket.SHUT_RDWR)
except Exception:
    pass
eth_sock.close()
print('Ethernet connection closed.')
